# London Housing Price Prediction

This notebook aims to predict housing prices in London. We will go through the steps of data loading, cleaning, feature engineering, model training, and prediction.

In [77]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Data Loading

First, we load the training, testing, and sample submission datasets into pandas DataFrames.

In [78]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')
df_sub = pd.read_csv('sample_submission.csv')

## 2. Initial Data Inspection and Preprocessing

We start by inspecting the datasets for missing values and general information. We'll also drop irrelevant columns.

### Missing Value Summary (Training Data)

Let's first inspect the count of missing values for each column in the training dataset to understand the scope of the problem.

In [79]:
df_train.isna().sum()

,0
ID,0
fullAddress,0
postcode,0
country,0
outcode,0
latitude,0
longitude,0
bathrooms,48479
bedrooms,24843
floorAreaSqM,13806


The output above shows columns with missing values and their respective counts in the training dataset.

### Dropping Irrelevant Columns

The `ID`, `fullAddress`, `country`, and `postcode` columns are either unique identifiers, contain too much specific information, or are not directly useful for our model, so we will drop them. The `outcode` column will be kept for one-hot encoding later.

In [80]:
df_train = df_train.drop(['ID', 'fullAddress', 'country', 'postcode'], axis = 1)
df_test = df_test.drop(['ID', 'fullAddress', 'country', 'postcode'], axis = 1)

### Inspecting Data Types and Non-Null Counts

We check the data types and non-null counts for both the training and test sets to understand the completeness of each feature.

### Missing Value Summary (Test Data)

Similarly, we inspect the count of missing values for each column in the test dataset.

In [81]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266325 entries, 0 to 266324
Data columns (total 13 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   outcode              266325 non-null  object 
 1   latitude             266325 non-null  float64
 2   longitude            266325 non-null  float64
 3   bathrooms            217846 non-null  float64
 4   bedrooms             241482 non-null  float64
 5   floorAreaSqM         252519 non-null  float64
 6   livingRooms          229285 non-null  float64
 7   tenure               260604 non-null  object 
 8   propertyType         265817 non-null  object 
 9   currentEnergyRating  209511 non-null  object 
 10  sale_month           266325 non-null  int64  
 11  sale_year            266325 non-null  int64  
 12  price                266325 non-null  int64  
dtypes: float64(6), int64(3), object(4)
memory usage: 26.4+ MB


In [82]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16547 entries, 0 to 16546
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   outcode              16547 non-null  object 
 1   latitude             16547 non-null  float64
 2   longitude            16547 non-null  float64
 3   bathrooms            13923 non-null  float64
 4   bedrooms             15172 non-null  float64
 5   floorAreaSqM         14541 non-null  float64
 6   livingRooms          14452 non-null  float64
 7   tenure               15957 non-null  object 
 8   propertyType         16380 non-null  object 
 9   currentEnergyRating  15050 non-null  object 
 10  sale_month           16547 non-null  int64  
 11  sale_year            16547 non-null  int64  
dtypes: float64(6), int64(2), object(4)
memory usage: 1.5+ MB


In [83]:
df_test.isna().sum()

,0
outcode,0
latitude,0
longitude,0
bathrooms,2624
bedrooms,1375
floorAreaSqM,2006
livingRooms,2095
tenure,590
propertyType,167
currentEnergyRating,1497


The test dataset also contains missing values across several features, which will be handled similarly to the training data.

### Handling Missing Values - `bathrooms`

We examine the distribution of `bathrooms` and impute missing values using random sampling based on the observed distribution. This is done for both training and testing datasets.

In [84]:
df_train.isna().sum()

,0
outcode,0
latitude,0
longitude,0
bathrooms,48479
bedrooms,24843
floorAreaSqM,13806
livingRooms,37040
tenure,5721
propertyType,508
currentEnergyRating,56814


In [85]:
df_train['bathrooms'].value_counts()

,count
bathrooms,
1.0,143432
2.0,58789
3.0,11786
4.0,2662
5.0,706
6.0,321
7.0,103
8.0,33
9.0,14


In [86]:
df_test['bathrooms'].value_counts()

,count
bathrooms,
1.0,8959
2.0,3944
3.0,761
4.0,170
5.0,57
6.0,23
7.0,5
8.0,3
9.0,1


Missing values in `bathrooms` have been imputed.

In [87]:
df_train.loc[df_train['bathrooms'].isna(), 'bathrooms'] = np.random.choice(a = [1.0, 2.0, 3.0, 4.0], size = df_train['bathrooms'].isna().sum(), p = [0.5, 0.3, 0.1, 0.1])
df_test.loc[df_test['bathrooms'].isna(), 'bathrooms'] = np.random.choice(a = [1.0, 2.0, 3.0, 4.0], size = df_test['bathrooms'].isna().sum(), p = [0.5, 0.3, 0.1, 0.1])

Missing values in `bathrooms` have been imputed using random sampling based on their observed distribution in the dataset.

### Handling Missing Values - `bedrooms`

Similarly, we examine the `bedrooms` distribution and impute missing values using random sampling.

In [88]:
df_train['bedrooms'].value_counts()

,count
bedrooms,
2.0,90652
3.0,61396
1.0,46794
4.0,27638
5.0,11222
6.0,2940
7.0,595
8.0,185
9.0,60


In [89]:
df_test['bedrooms'].value_counts()

,count
bedrooms,
2.0,5562
3.0,3996
1.0,2863
4.0,1803
5.0,686
6.0,194
7.0,50
8.0,13
9.0,5


Missing values in `bedrooms` have been imputed.

In [90]:
df_train.loc[df_train['bedrooms'].isna(), 'bedrooms'] = np.random.choice(a = [2.0, 3.0, 1.0, 4.0], size = df_train['bedrooms'].isna().sum(), p = [0.5, 0.3, 0.1, 0.1])
df_test.loc[df_test['bedrooms'].isna(), 'bedrooms'] = np.random.choice(a = [2.0, 3.0, 1.0, 4.0], size = df_test['bedrooms'].isna().sum(), p = [0.5, 0.3, 0.1, 0.1])

Missing values in `bedrooms` have been imputed using random sampling based on their observed distribution in the dataset.

### Handling Missing Values - `floorAreaSqM` and `livingRooms`

For `floorAreaSqM` and `livingRooms`, we first create indicator columns to mark where values were originally missing. Then, we impute `livingRooms` with the most frequent value (1.0). For `floorAreaSqM`, we impute missing values based on the median `floorAreaSqM` for each `livingRooms` group. If there are still missing values (e.g., if a `livingRooms` category had no non-null `floorAreaSqM`), we fill them with the overall median `floorAreaSqM`.

In [91]:
df_train['floorAreaSqM'].value_counts()

,count
floorAreaSqM,
55.0,4477
70.0,3911
56.0,3761
80.0,3512
54.0,3454
...,...
490.0,2
469.0,1
421.0,1


In [92]:
df_test['floorAreaSqM'].value_counts()

,count
floorAreaSqM,
55.0,286
70.0,238
54.0,220
56.0,218
80.0,209
...,...
308.0,1
448.0,1
403.0,1


In [93]:
df_test['floorAreaSqM'].value_counts()

,count
floorAreaSqM,
55.0,286
70.0,238
54.0,220
56.0,218
80.0,209
...,...
308.0,1
448.0,1
403.0,1


In [94]:
df_train['floorAreaSqM'].value_counts()

,count
floorAreaSqM,
55.0,4477
70.0,3911
56.0,3761
80.0,3512
54.0,3454
...,...
490.0,2
469.0,1
421.0,1


In [95]:
len(df_train['floorAreaSqM'])

266325

In [96]:
df_train['livingRooms'].value_counts()

,count
livingRooms,
1.0,174397
2.0,45182
3.0,7891
4.0,1380
5.0,328
6.0,76
7.0,26
8.0,4
9.0,1


In [97]:
len(df_train['livingRooms'])

266325

In [98]:
df_test.isna().sum()

,0
outcode,0
latitude,0
longitude,0
bathrooms,0
bedrooms,0
floorAreaSqM,2006
livingRooms,2095
tenure,590
propertyType,167
currentEnergyRating,1497


In [99]:
df_train.isna().sum()

,0
outcode,0
latitude,0
longitude,0
bathrooms,0
bedrooms,0
floorAreaSqM,13806
livingRooms,37040
tenure,5721
propertyType,508
currentEnergyRating,56814


Creating indicator columns for missing `livingRooms` and `floorAreaSqM`.

In [100]:
df_train['rooms_is_missing'] = df_train['livingRooms'].isna().astype(int)
df_train['area_is_missing'] = df_train['floorAreaSqM'].isna().astype(int)

Imputing missing `livingRooms` with 1.0 and `floorAreaSqM` with median grouped by `livingRooms` for the training set.

In [101]:
df_train['livingRooms'] = df_train['livingRooms'].fillna(1.0)

df_train['floorAreaSqM'] = df_train['floorAreaSqM'].fillna(df_train.groupby('livingRooms')['floorAreaSqM'].transform('median'))

Applying similar imputation steps for the test set.

In [102]:
df_test['rooms_is_missing'] = df_test['livingRooms'].isna().astype(int)
df_test['area_is_missing'] = df_test['floorAreaSqM'].isna().astype(int)

In [103]:
df_test['livingRooms'] = df_test['livingRooms'].fillna(1.0)

In [104]:
df_test['floorAreaSqM'] = df_test['floorAreaSqM'].fillna(df_test.groupby('livingRooms')['floorAreaSqM'].transform('median'))

Final imputation for any remaining `floorAreaSqM` missing values in the training set using the overall median.

In [105]:
print(df_train[df_train['floorAreaSqM'].isna()]['livingRooms'])

4925      8.0
19711     8.0
43856     8.0
182691    9.0
258245    8.0
Name: livingRooms, dtype: float64


In [106]:
df_train['floorAreaSqM'] = df_train['floorAreaSqM'].fillna(df_train['floorAreaSqM'].median())

Missing values for `floorAreaSqM` and `livingRooms` have been successfully imputed. Indicator columns were created to preserve information about the original missingness.

### Handling Missing Values - `tenure`

We examine the `tenure` distribution and impute missing values by randomly sampling 'Leasehold' or 'Freehold' based on their proportions.

In [107]:
df_train['propertyType'].value_counts()

,count
propertyType,
Purpose Built Flat,68726
Flat/Maisonette,61139
Mid Terrace House,45649
Converted Flat,32552
Semi-Detached House,20475
Terrace Property,15114
End Terrace House,13063
Detached House,6666
Terraced,927


In [108]:
len(df_train['tenure'])

266325

In [109]:
df_test['tenure'].value_counts()

,count
tenure,
Leasehold,9017
Freehold,6643
Feudal,229
Shared,68


Missing values in `tenure` have been imputed.

In [110]:
df_train.loc[df_train['tenure'].isna(), 'tenure'] = np.random.choice(a = ['Leasehold', 'Freehold'], size = df_train['tenure'].isna().sum(), p = [0.7, 0.3])
df_test.loc[df_test['tenure'].isna(), 'tenure'] = np.random.choice(a = ['Leasehold', 'Freehold'], size = df_test['tenure'].isna().sum(), p = [0.7, 0.3])

Missing values in `tenure` have been imputed by randomly sampling 'Leasehold' or 'Freehold' based on their proportions.

In [111]:
df_train['tenure'].value_counts()

,count
tenure,
Leasehold,159946
Freehold,103781
Feudal,1906
Shared,692


### Handling Missing Values - `propertyType`

Missing values in `propertyType` are imputed with 'Purpose Built Flat', as it is a common type.

In [112]:
df_train['propertyType'] = df_train['propertyType'].fillna('Purpose Built Flat')
df_test['propertyType'] = df_test['propertyType'].fillna('Purpose Built Flat')

Missing values in `propertyType` have been filled with 'Purpose Built Flat', which is the most frequent category.

### Handling Missing Values - `currentEnergyRating`

We create an indicator column for missing `currentEnergyRating` values, impute missing values with 'D' (the most frequent rating), and then map the categorical ratings to numerical values.

In [113]:
df_train['currentEnergyRating'].value_counts()

,count
currentEnergyRating,
D,87925
C,78356
B,20836
E,20253
F,1519
G,436
A,186


In [114]:
df_test['currentEnergyRating'].value_counts()

,count
currentEnergyRating,
D,5829
C,5596
B,2037
E,1374
F,127
G,67
A,20


In [115]:
df_test.isna().sum()

,0
outcode,0
latitude,0
longitude,0
bathrooms,0
bedrooms,0
floorAreaSqM,0
livingRooms,0
tenure,0
propertyType,0
currentEnergyRating,1497


Creating indicator column for missing `currentEnergyRating` and imputing for the training set.

In [116]:
df_train['energy_is_missing'] = df_train['currentEnergyRating'].isna().astype(int)

In [117]:
df_train['currentEnergyRating'] = df_train['currentEnergyRating'].fillna('D')

Defining a dictionary to map energy ratings to numerical values.

In [118]:
rating_dict = {'A': 7, 'B': 6, 'C': 5, 'D': 4, 'E': 3, 'F': 2, 'G': 1}

Mapping energy ratings to numerical values for the training set.

In [119]:
df_train['currentEnergyRating'] = df_train['currentEnergyRating'].map(rating_dict)

Applying the same steps for the test set.

In [120]:
df_test['energy_is_missing'] = df_test['currentEnergyRating'].isna().astype(int)
df_test['currentEnergyRating'] = df_test['currentEnergyRating'].fillna('D')
df_test['currentEnergyRating'] = df_test['currentEnergyRating'].map(rating_dict)

### Verifying Data Cleaning

We check `df.info()` again to ensure all missing values have been handled and data types are appropriate.

In [121]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 266325 entries, 0 to 266324
Data columns (total 16 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   outcode              266325 non-null  object 
 1   latitude             266325 non-null  float64
 2   longitude            266325 non-null  float64
 3   bathrooms            266325 non-null  float64
 4   bedrooms             266325 non-null  float64
 5   floorAreaSqM         266325 non-null  float64
 6   livingRooms          266325 non-null  float64
 7   tenure               266325 non-null  object 
 8   propertyType         266325 non-null  object 
 9   currentEnergyRating  266325 non-null  int64  
 10  sale_month           266325 non-null  int64  
 11  sale_year            266325 non-null  int64  
 12  price                266325 non-null  int64  
 13  rooms_is_missing     266325 non-null  int64  
 14  area_is_missing      266325 non-null  int64  
 15  energy_is_missing

In [122]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16547 entries, 0 to 16546
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   outcode              16547 non-null  object 
 1   latitude             16547 non-null  float64
 2   longitude            16547 non-null  float64
 3   bathrooms            16547 non-null  float64
 4   bedrooms             16547 non-null  float64
 5   floorAreaSqM         16547 non-null  float64
 6   livingRooms          16547 non-null  float64
 7   tenure               16547 non-null  object 
 8   propertyType         16547 non-null  object 
 9   currentEnergyRating  16547 non-null  int64  
 10  sale_month           16547 non-null  int64  
 11  sale_year            16547 non-null  int64  
 12  rooms_is_missing     16547 non-null  int64  
 13  area_is_missing      16547 non-null  int64  
 14  energy_is_missing    16547 non-null  int64  
dtypes: float64(6), int64(6), object(3)
m

## 3. Feature Engineering

We use one-hot encoding for categorical features like `outcode`, `tenure`, and `propertyType` to convert them into a numerical format suitable for machine learning models.

In [123]:
df_train['outcode'].value_counts()

,count
outcode,
SE18,4444
SW2,4440
N16,4163
SW4,4102
SW16,4006
...,...
EC3V,17
EC2R,7
EC2V,7


In [124]:
df_train = pd.get_dummies(df_train, columns=['outcode', 'tenure', 'propertyType'], drop_first=True, dtype=int)
df_test = pd.get_dummies(df_test, columns=['outcode', 'tenure', 'propertyType'], drop_first=True, dtype=int)

One-hot encoding has been applied to `outcode`, `tenure`, and `propertyType` to transform these categorical features into a numerical format suitable for machine learning models. `drop_first=True` is used to avoid multicollinearity.

Aligning columns between `df_train` and `df_test` after one-hot encoding to ensure they have the same features. `fill_value=0` handles columns present in one but not the other.

In [125]:
df_train, df_test = df_train.align(df_test, join='left', axis=1, fill_value=0)

## 4. Model Preparation

We import necessary libraries for scaling and model building. Then, we separate the target variable (`price`) from the features in the training set and prepare the test set.

In [126]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import SGDRegressor

Splitting data into training features (X_train), training target (y_train), and test features (X_test).

In [127]:
print(df_train.columns.tolist())

['latitude', 'longitude', 'bathrooms', 'bedrooms', 'floorAreaSqM', 'livingRooms', 'currentEnergyRating', 'sale_month', 'sale_year', 'price', 'rooms_is_missing', 'area_is_missing', 'energy_is_missing', 'outcode_E10', 'outcode_E11', 'outcode_E12', 'outcode_E13', 'outcode_E14', 'outcode_E15', 'outcode_E16', 'outcode_E17', 'outcode_E18', 'outcode_E1W', 'outcode_E2', 'outcode_E3', 'outcode_E4', 'outcode_E5', 'outcode_E6', 'outcode_E7', 'outcode_E8', 'outcode_E9', 'outcode_EC1A', 'outcode_EC1M', 'outcode_EC1N', 'outcode_EC1R', 'outcode_EC1V', 'outcode_EC1Y', 'outcode_EC2A', 'outcode_EC2M', 'outcode_EC2R', 'outcode_EC2V', 'outcode_EC2Y', 'outcode_EC3A', 'outcode_EC3M', 'outcode_EC3N', 'outcode_EC3R', 'outcode_EC3V', 'outcode_EC4A', 'outcode_EC4M', 'outcode_EC4R', 'outcode_EC4V', 'outcode_EC4Y', 'outcode_N1', 'outcode_N10', 'outcode_N11', 'outcode_N12', 'outcode_N13', 'outcode_N14', 'outcode_N15', 'outcode_N16', 'outcode_N17', 'outcode_N18', 'outcode_N19', 'outcode_N2', 'outcode_N20', 'outcode

In [128]:
print(df_test.columns.tolist())

['latitude', 'longitude', 'bathrooms', 'bedrooms', 'floorAreaSqM', 'livingRooms', 'currentEnergyRating', 'sale_month', 'sale_year', 'price', 'rooms_is_missing', 'area_is_missing', 'energy_is_missing', 'outcode_E10', 'outcode_E11', 'outcode_E12', 'outcode_E13', 'outcode_E14', 'outcode_E15', 'outcode_E16', 'outcode_E17', 'outcode_E18', 'outcode_E1W', 'outcode_E2', 'outcode_E3', 'outcode_E4', 'outcode_E5', 'outcode_E6', 'outcode_E7', 'outcode_E8', 'outcode_E9', 'outcode_EC1A', 'outcode_EC1M', 'outcode_EC1N', 'outcode_EC1R', 'outcode_EC1V', 'outcode_EC1Y', 'outcode_EC2A', 'outcode_EC2M', 'outcode_EC2R', 'outcode_EC2V', 'outcode_EC2Y', 'outcode_EC3A', 'outcode_EC3M', 'outcode_EC3N', 'outcode_EC3R', 'outcode_EC3V', 'outcode_EC4A', 'outcode_EC4M', 'outcode_EC4R', 'outcode_EC4V', 'outcode_EC4Y', 'outcode_N1', 'outcode_N10', 'outcode_N11', 'outcode_N12', 'outcode_N13', 'outcode_N14', 'outcode_N15', 'outcode_N16', 'outcode_N17', 'outcode_N18', 'outcode_N19', 'outcode_N2', 'outcode_N20', 'outcode

In [129]:
X_train = df_train.drop('price', axis=1)
y_train = df_train['price']
X_test = df_test.copy()

## 5. Model Definition and Training

We define a `RobustScaler` to handle outliers and an `SGDRegressor` for linear regression. These are combined into a pipeline for cleaner workflow. We then perform `GridSearchCV` to find the best hyperparameters for our `SGDRegressor`.

In [130]:
scaler = RobustScaler()

### Setting up the Model Pipeline

We'll use a `Pipeline` to chain a `RobustScaler` (to handle potential outliers in features) and an `SGDRegressor` (a linear model suitable for large datasets). This ensures consistent preprocessing for both training and prediction, and helps prevent data leakage.

In [131]:
model = SGDRegressor()

In [132]:
pipe = Pipeline([('scaler', scaler), ('model', model)])

Defining the hyperparameter grid for `GridSearchCV`.

In [133]:
from sklearn.model_selection import GridSearchCV

In [134]:
param_grid = {
    'model__penalty': ['l2', 'l1'],
    'model__alpha': [0.0001, 0.1],
    'model__learning_rate': ['invscaling', 'adaptive']
}

Initializing `GridSearchCV` with the pipeline and parameter grid. Cross-validation (cv=3) and verbose output are set.

In [135]:
final_model = GridSearchCV(pipe, param_grid, cv=3, verbose=2)

Fitting the `GridSearchCV` to the training data to find the best model parameters.

In [136]:
final_model.fit(X_train, y_train)

Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END model__alpha=0.0001, model__learning_rate=invscaling, model__penalty=l2; total time=  14.1s
[CV] END model__alpha=0.0001, model__learning_rate=invscaling, model__penalty=l2; total time=  14.2s
[CV] END model__alpha=0.0001, model__learning_rate=invscaling, model__penalty=l2; total time=  11.8s
[CV] END model__alpha=0.0001, model__learning_rate=invscaling, model__penalty=l1; total time=  22.6s
[CV] END model__alpha=0.0001, model__learning_rate=invscaling, model__penalty=l1; total time=  24.9s
[CV] END model__alpha=0.0001, model__learning_rate=invscaling, model__penalty=l1; total time=  13.2s
[CV] END model__alpha=0.0001, model__learning_rate=adaptive, model__penalty=l2; total time=  20.0s
[CV] END model__alpha=0.0001, model__learning_rate=adaptive, model__penalty=l2; total time=  17.1s
[CV] END model__alpha=0.0001, model__learning_rate=adaptive, model__penalty=l2; total time=  16.7s
[CV] END model__alpha=0.0001, model__

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('scaler', RobustScaler()),
                                       ('model', SGDRegressor())]),
             param_grid={'model__alpha': [0.0001, 0.1],
                         'model__learning_rate': ['invscaling', 'adaptive'],
                         'model__penalty': ['l2', 'l1']},
             verbose=2)

## 6. Model Evaluation
  After training, we evaluate the model's performance using metrics like Root Mean Squared Error (RMSE) to understand how well it predicts house prices. A lower RMSE indicates a better fit of the model to the data.

In [137]:
from sklearn.metrics import mean_squared_error

### Evaluating Training Set Performance

We evaluate the model's performance on the training set using Root Mean Squared Error (RMSE) and Mean Absolute Error (MAE). These metrics provide a measure of the average magnitude of the errors, with MAE being less sensitive to outliers than RMSE.

In [138]:
y_train_pred = final_model.predict(X_train)

In [139]:
from sklearn.metrics import mean_absolute_error
print(f"MAE = : {mean_absolute_error(y_train, y_train_pred):,.2f}")

MAE = : 317,272.05


In [147]:
print(f"RMSE = : {np.sqrt(mean_squared_error(y_train, y_train_pred)):,.2f}")

RMSE = : 1,125,580.07


The `X_test` dataframe still contains the `price` column from the `align` operation. This column needs to be dropped before prediction to match the features used for training.

### Preparing Test Data for Prediction

Before making predictions on the test set, we need to ensure that the `X_test` DataFrame has the exact same columns as the `X_train` DataFrame used for training. The `price` column, which was added during the alignment operation, must be dropped as it is the target variable.

In [140]:
X_test = X_test.drop('price', axis=1)

## 7. Prediction and Submission

After training the model, we use it to make predictions on the test set and prepare the submission file.

In [141]:
y_pred = final_model.predict(X_test)

Assigning the predicted prices to the `price` column in the submission DataFrame.

In [142]:
df_sub['price'] = y_pred

Displaying the first few rows of the submission DataFrame.

In [143]:
df_sub

,ID,price
0,266325,5.248052e+05
1,266326,4.123449e+05
2,266327,5.069206e+05
3,266328,5.945169e+05
4,266329,7.495450e+05
...,...,...
16542,282867,1.631511e+05
16543,282868,2.610550e+05
16544,282869,5.394561e+05
16545,282870,1.042041e+06


Saving the submission DataFrame to a CSV file named 'sub.csv'.

In [144]:
df_sub.to_csv('sub_baseline_sdg.csv', index=False, index_label = False)